In [ ]:
import numpy as np
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from utils.data import load_and_process_data, preprocess_data
from utils.model import create_model

In [ ]:
DATA_DIR = "/root/tmp/PulseGUARD/classification" 
TEST_SIZE = 0.2
RANDOM_STATE = 42

# Loading and processing data
ecg_data, labels = load_and_process_data(DATA_DIR)

# Performed oversampling using SMOTE
smote = SMOTE(random_state=RANDOM_STATE)
ecg_data_reshaped = ecg_data.reshape(ecg_data.shape[0], -1)  
ecg_data_oversampled, labels_oversampled = smote.fit_resample(ecg_data_reshaped, labels)
ecg_data_oversampled = ecg_data_oversampled.reshape(-1, 100)  
    
# Convert the labels to one-hot encoding
X, y = preprocess_data(ecg_data_oversampled, labels_oversampled)
    
# Divide the training set/test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
    
# Calculate the category weights
class_weights = compute_class_weight('balanced', classes=np.unique(labels_oversampled), y=labels_oversampled)
class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}
    
# Create a model
input_shape = (100, 1)  
model = create_model(input_shape, y.shape[1])
    
#  Training the Model
history = model.fit(X_train, y_train,
                    epochs=50,
                    batch_size=32,
                    validation_split=0.2,
                    verbose=1,
                    class_weight=class_weights_dict)  

In [ ]:


model.save('/root/tmp/PulseGUARD/model/classification_model.h5')

del model  

In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model("/root/tmp/PulseGUARD/model/classification_model.h5")
print(model) 


In [ ]:
from sklearn.metrics import classification_report

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")


print("prediction...")
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)
    

print("Classification report:")
print(classification_report(y_true_classes, y_pred_classes))

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("Generating confusion matrix...")
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)
    
unique_labels = np.unique(np.concatenate((y_true_classes, y_pred_classes)))
class_labels = [str(label) for label in unique_labels]
    

y_pred_prob = model.predict(X_test)


y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)  


cm = confusion_matrix(y_true, y_pred)


class_names =['AFIB', 'AFLT', 'MI', 'NORM', 'PSVT', 'PVC']

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=class_names, 
            yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=class_names, 
            yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Normalized Confusion Matrix")
plt.show()